# 06 - Benchmark Unificado: EfficientNet vs U-Net Multi-task

Este notebook substitui os notebooks `04_classification.ipynb` e `05_unet_classification.ipynb` por um fluxo unico de benchmark.

**Objetivo**

Comparar dois modelos em tres datasets diferentes:

- `with_augmentation_64x64`
- `with_augmentation_224x224`
- `without_augmentation`

Os dois modelos avaliados sao:

- `EfficientNet-B0`: modelo de classificacao usado no notebook 4
- `U-Net Multi-task`: modelo de segmentacao + classificacao adaptado do notebook 5

**Criterio principal de comparacao**

Como o objetivo do projeto eh **maximizar a deteccao de melanomas** e **reduzir falsos negativos**, o ranking final prioriza:

1. sensibilidade no test set
2. numero de falsos negativos
3. AUC
4. especificidade

Para isso, o threshold de decisao eh escolhido no conjunto de validacao com foco em **alta sensibilidade**.
        

In [1]:
import json
import math
import random
import warnings
from functools import lru_cache
from pathlib import Path

import cv2
import matplotlib
from IPython import get_ipython

ipython = get_ipython()
if ipython is None or ipython.__class__.__name__ != "ZMQInteractiveShell":
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from sklearn.metrics import auc, confusion_matrix, f1_score, roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

print(f"torch version   : {torch.__version__}")
print(f"timm version    : {timm.__version__}")
print(f"CUDA disponivel : {torch.cuda.is_available()}")
print(f"MPS disponivel  : {torch.backends.mps.is_available()}")
        

/Users/eduardoyaginuma/Documents/Repositorios/insper/skin-cancer-images-segmentation/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch version   : 2.11.0
timm version    : 1.0.26
CUDA disponivel : False
MPS disponivel  : True


## Configuracao

O notebook resolve automaticamente os datasets existentes em `data/processed/` e usa explicitamente as branches `64x64`, `224x224` e `without_augmentation`.
        

In [2]:
def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for candidate in candidates:
        if (candidate / "data" / "metadata.csv").exists() and (candidate / "docs").exists():
            return candidate
    raise FileNotFoundError("Nao foi possivel localizar a raiz com data/metadata.csv.")


ROOT_DIR = resolve_repo_root()
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MASK_DIR = DATA_DIR / "masks"
PREP_DIR = ROOT_DIR / "notebooks" / "outputs" / "preprocessing"
FIG_DIR = ROOT_DIR / "outputs" / "figures" / "model_comparison"
MODEL_DIR = ROOT_DIR / "outputs" / "models" / "model_comparison"

for directory in [FIG_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

PIN_MEMORY = DEVICE.type == "cuda"

REQUESTED_DATASETS = [
    {
        "requested_name": "with_augmentation_64x64",
        "aliases": ["with_augmentation_64x64"],
        "family": "augmented",
    },
    {
        "requested_name": "with_augmentation_224x224",
        "aliases": ["with_augmentation_224x224", "with_augmentation"],
        "family": "augmented",
    },
    {
        "requested_name": "without_augmentation",
        "aliases": ["without_augmentation"],
        "family": "baseline",
    },
]

SELECTED_DATASET_NAMES = None

MELANOMA_RECALL_TARGET = 0.95
HARD_NEGATIVE_LABEL = "NV"
HARD_NEGATIVE_MULTIPLIER = 2.0

EFFICIENTNET_MODEL_NAME = "efficientnet_b0"
USE_IMAGENET_PRETRAINING = True
EFFICIENTNET_LR = 1e-4
EFFICIENTNET_WEIGHT_DECAY = 1e-4
EFFICIENTNET_EPOCHS = 16
EFFICIENTNET_PATIENCE = 4

UNET_LR = 1e-4
UNET_WEIGHT_DECAY = 1e-4
UNET_EPOCHS = 18
UNET_PATIENCE = 5
UNET_SEG_LOSS_WEIGHT = 0.4
UNET_ENCODER_FILTERS = [64, 128, 256, 512]
UNET_BRIDGE_FILTERS = 1024

BATCH_SIZE_CONFIG = {
    "efficientnet": {64: 32, 128: 24, 224: 16, "default": 16},
    "unet_multitask": {64: 16, 128: 12, 224: 8, "default": 8},
}

BBOX_MARGIN_RATIO = 0.10
MASK_THRESHOLD = 127

config = {
    "seed": SEED,
    "device": str(DEVICE),
    "selected_dataset_names": SELECTED_DATASET_NAMES,
    "melanoma_recall_target": MELANOMA_RECALL_TARGET,
    "efficientnet_model_name": EFFICIENTNET_MODEL_NAME,
    "use_imagenet_pretraining": USE_IMAGENET_PRETRAINING,
    "efficientnet_epochs": EFFICIENTNET_EPOCHS,
    "unet_epochs": UNET_EPOCHS,
    "unet_seg_loss_weight": UNET_SEG_LOSS_WEIGHT,
    "requested_datasets": REQUESTED_DATASETS,
}
print(json.dumps(config, indent=2))
        

{
  "seed": 42,
  "device": "mps",
  "selected_dataset_names": null,
  "melanoma_recall_target": 0.95,
  "efficientnet_model_name": "efficientnet_b0",
  "use_imagenet_pretraining": true,
  "efficientnet_epochs": 16,
  "unet_epochs": 18,
  "unet_seg_loss_weight": 0.4,
  "requested_datasets": [
    {
      "requested_name": "with_augmentation_64x64",
      "aliases": [
        "with_augmentation_64x64"
      ],
      "family": "augmented"
    },
    {
      "requested_name": "with_augmentation_224x224",
      "aliases": [
        "with_augmentation_224x224",
        "with_augmentation"
      ],
      "family": "augmented"
    },
    {
      "requested_name": "without_augmentation",
      "aliases": [
        "without_augmentation"
      ],
      "family": "baseline"
    }
  ]
}


## 1. Resolucao dos datasets

Esta etapa:

- resolve os aliases de dataset
- carrega manifests de `train/val/test`
- detecta automaticamente o tamanho real das imagens
- carrega ou recalcula as estatisticas de normalizacao
        

In [3]:
def get_batch_size(model_family: str, image_size: int) -> int:
    mapping = BATCH_SIZE_CONFIG[model_family]
    return mapping.get(image_size, mapping["default"])


def infer_image_size_from_manifest(manifest: pd.DataFrame) -> int:
    sample_path = Path(manifest.iloc[0]["export_path"])
    with Image.open(sample_path).convert("RGB") as image:
        width, height = image.size
    if width != height:
        raise ValueError(f"Imagem nao quadrada encontrada em {sample_path}: {(width, height)}")
    return int(width)


def resolve_dataset_root(spec: dict) -> tuple[str, Path]:
    for alias in spec["aliases"]:
        candidate = PROCESSED_DIR / alias
        if candidate.exists():
            return alias, candidate
    raise FileNotFoundError(
        f"Nenhum dataset encontrado para {spec['requested_name']}. "
        f"Aliases tentados: {spec['aliases']}"
    )


def compute_norm_stats_from_manifest(train_manifest: pd.DataFrame, sample_size: int = 512) -> dict:
    usable_manifest = train_manifest.copy()
    if "is_augmented" in usable_manifest.columns:
        original_mask = ~usable_manifest["is_augmented"].astype(bool)
        if int(original_mask.sum()) > 0:
            usable_manifest = usable_manifest.loc[original_mask].reset_index(drop=True)

    sample = usable_manifest.sample(n=min(sample_size, len(usable_manifest)), random_state=SEED)

    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sq_sum = np.zeros(3, dtype=np.float64)
    pixel_count = 0

    for row in sample.itertuples():
        image = np.array(Image.open(row.export_path).convert("RGB"), dtype=np.float32) / 255.0
        pixels = image.reshape(-1, 3)
        channel_sum += pixels.sum(axis=0)
        channel_sq_sum += (pixels ** 2).sum(axis=0)
        pixel_count += len(pixels)

    mean = channel_sum / pixel_count
    std = np.sqrt(np.maximum(channel_sq_sum / pixel_count - mean ** 2, 1e-12))

    return {
        "mean": mean.round(6).tolist(),
        "std": std.round(6).tolist(),
        "sample_count": int(len(sample)),
    }


def load_or_compute_norm_stats(image_size: int, train_manifest: pd.DataFrame) -> dict:
    candidate_paths = [
        PREP_DIR / f"normalization_stats_{image_size}x{image_size}.json",
    ]
    if image_size == 224:
        candidate_paths.insert(0, PREP_DIR / "normalization_stats.json")

    for path in candidate_paths:
        if path.exists():
            stats = json.loads(path.read_text())
            return {
                "mean": stats["mean"],
                "std": stats["std"],
                "source": str(path),
            }

    stats = compute_norm_stats_from_manifest(train_manifest)
    cache_path = PREP_DIR / f"normalization_stats_autogen_{image_size}x{image_size}.json"
    cache_path.write_text(json.dumps(stats, indent=2), encoding="utf-8")
    return {
        "mean": stats["mean"],
        "std": stats["std"],
        "source": str(cache_path),
    }


dataset_registry = {}
dataset_rows = []

for spec in REQUESTED_DATASETS:
    if SELECTED_DATASET_NAMES is not None and spec["requested_name"] not in SELECTED_DATASET_NAMES:
        continue

    resolved_name, dataset_root = resolve_dataset_root(spec)
    manifests = {
        split: pd.read_csv(dataset_root / f"{split}_manifest.csv")
        for split in ["train", "val", "test"]
    }
    image_size = infer_image_size_from_manifest(manifests["train"])
    norm_stats = load_or_compute_norm_stats(image_size, manifests["train"])

    dataset_info = {
        "requested_name": spec["requested_name"],
        "resolved_name": resolved_name,
        "dataset_root": dataset_root,
        "family": spec["family"],
        "image_size": image_size,
        "norm_mean": norm_stats["mean"],
        "norm_std": norm_stats["std"],
        "norm_source": norm_stats["source"],
        "manifests": manifests,
        "has_augmented_train": bool(manifests["train"]["is_augmented"].any()) if "is_augmented" in manifests["train"].columns else False,
    }

    dataset_key = spec["requested_name"]
    dataset_registry[dataset_key] = dataset_info

    train_manifest = manifests["train"]
    dataset_rows.append(
        {
            "dataset_key": dataset_key,
            "resolved_name": resolved_name,
            "image_size": image_size,
            "train_images": len(train_manifest),
            "train_originals": int((~train_manifest["is_augmented"]).sum()) if "is_augmented" in train_manifest.columns else len(train_manifest),
            "train_augmented": int(train_manifest["is_augmented"].sum()) if "is_augmented" in train_manifest.columns else 0,
            "val_images": len(manifests["val"]),
            "test_images": len(manifests["test"]),
            "norm_source": norm_stats["source"],
        }
    )

dataset_summary = pd.DataFrame(dataset_rows)
display(dataset_summary)
        

,dataset_key,resolved_name,image_size,train_images,train_originals,train_augmented,val_images,test_images,norm_source
0,with_augmentation_64x64,with_augmentation_64x64,64,9348,3116,6232,668,668,/Users/eduardoyaginuma/Documents/Repositorios/...
1,with_augmentation_224x224,with_augmentation_224x224,224,9348,3116,6232,668,668,/Users/eduardoyaginuma/Documents/Repositorios/...
2,without_augmentation,without_augmentation,224,3116,3116,0,668,668,/Users/eduardoyaginuma/Documents/Repositorios/...


## 2. Funcoes compartilhadas de avaliacao

O threshold final nao usa diretamente o ponto de Youden. Em vez disso, ele eh escolhido na validacao com prioridade para **alta sensibilidade**, que eh a metrica mais importante para o problema clinico.
        

In [4]:
def compute_binary_metrics(
    probs: list[float] | np.ndarray,
    labels: list[int] | np.ndarray,
    threshold: float,
) -> dict:
    probs_arr = np.asarray(probs, dtype=np.float32)
    labels_arr = np.asarray(labels, dtype=np.int64)
    preds = (probs_arr >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(labels_arr, preds, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = f1_score(labels_arr, preds, zero_division=0)
    accuracy = (tp + tn) / max(len(labels_arr), 1)
    auc_score = roc_auc_score(labels_arr, probs_arr) if len(np.unique(labels_arr)) > 1 else 0.0

    beta_sq = 4.0
    if precision == 0.0 and sensitivity == 0.0:
        f2 = 0.0
    else:
        f2 = (1 + beta_sq) * precision * sensitivity / max(beta_sq * precision + sensitivity, 1e-12)

    return {
        "auc": float(auc_score),
        "threshold": float(threshold),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "precision": float(precision),
        "f1": float(f1),
        "f2": float(f2),
        "accuracy": float(accuracy),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "preds": preds,
        "labels": labels_arr,
        "probs": probs_arr,
    }


def select_threshold_for_high_recall(
    probs: list[float] | np.ndarray,
    labels: list[int] | np.ndarray,
    target_recall: float,
) -> dict:
    probs_arr = np.asarray(probs, dtype=np.float32)
    labels_arr = np.asarray(labels, dtype=np.int64)

    candidate_thresholds = np.unique(np.round(np.concatenate(([0.0], probs_arr, [1.0])), 6))
    candidate_thresholds = np.sort(candidate_thresholds)

    rows = []
    for threshold in candidate_thresholds:
        metrics = compute_binary_metrics(probs_arr, labels_arr, float(threshold))
        rows.append(
            {
                "threshold": metrics["threshold"],
                "sensitivity": metrics["sensitivity"],
                "specificity": metrics["specificity"],
                "precision": metrics["precision"],
                "f1": metrics["f1"],
                "f2": metrics["f2"],
                "fn": metrics["fn"],
                "fp": metrics["fp"],
            }
        )

    threshold_table = pd.DataFrame(rows)
    feasible = threshold_table[threshold_table["sensitivity"] >= target_recall].copy()

    if len(feasible) > 0:
        best_row = feasible.sort_values(
            ["specificity", "f2", "threshold"],
            ascending=[False, False, True],
        ).iloc[0]
        selection_rule = (
            f"Maior especificidade entre os thresholds com sensibilidade >= {target_recall:.2f} na validacao."
        )
        target_hit = True
    else:
        best_row = threshold_table.sort_values(
            ["sensitivity", "specificity", "threshold"],
            ascending=[False, False, True],
        ).iloc[0]
        selection_rule = (
            f"Nenhum threshold atingiu sensibilidade >= {target_recall:.2f}; "
            "foi escolhido o threshold com maior sensibilidade disponivel na validacao."
        )
        target_hit = False

    best_metrics = compute_binary_metrics(probs_arr, labels_arr, float(best_row["threshold"]))
    best_metrics["selection_rule"] = selection_rule
    best_metrics["target_recall"] = float(target_recall)
    best_metrics["target_hit"] = bool(target_hit)
    best_metrics["threshold_table"] = threshold_table
    return best_metrics


def compute_roc_payload(probs: list[float], labels: list[int]) -> dict:
    fpr, tpr, thresholds = roc_curve(labels, probs)
    return {
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "auc": float(auc(fpr, tpr)),
    }


def clear_device_cache() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("Funcoes compartilhadas de avaliacao definidas.")
        

Funcoes compartilhadas de avaliacao definidas.


## 3. Modelo A: EfficientNet-B0
        

In [5]:
class ManifestClassificationDataset(Dataset):
    def __init__(self, manifest: pd.DataFrame, image_size: int, mean: list[float], std: list[float]) -> None:
        self.records = manifest.reset_index(drop=True)
        self.transform = transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ]
        )

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.records.iloc[idx]
        image = Image.open(row["export_path"]).convert("RGB")
        tensor = self.transform(image)
        label = torch.tensor(int(row["binary_label"]), dtype=torch.float32)
        return tensor, label


def build_weighted_sampler(manifest: pd.DataFrame) -> WeightedRandomSampler:
    positive_count = max(int((manifest["binary_label"] == 1).sum()), 1)
    negative_count = max(int((manifest["binary_label"] == 0).sum()), 1)
    positive_weight = negative_count / positive_count

    weights = []
    for row in manifest.itertuples():
        weight = positive_weight if row.binary_label == 1 else 1.0
        if getattr(row, "label", None) == HARD_NEGATIVE_LABEL:
            weight *= HARD_NEGATIVE_MULTIPLIER
        weights.append(weight)

    return WeightedRandomSampler(
        weights=torch.DoubleTensor(weights),
        num_samples=len(weights),
        replacement=True,
    )


def build_classification_loaders(dataset_info: dict) -> dict:
    manifests = dataset_info["manifests"]
    image_size = dataset_info["image_size"]
    mean = dataset_info["norm_mean"]
    std = dataset_info["norm_std"]

    datasets_dict = {
        split: ManifestClassificationDataset(frame, image_size, mean, std)
        for split, frame in manifests.items()
    }

    train_manifest = manifests["train"]
    use_weighted_sampler = not dataset_info["has_augmented_train"]

    if use_weighted_sampler:
        train_sampler = build_weighted_sampler(train_manifest)
        train_loader = DataLoader(
            datasets_dict["train"],
            batch_size=get_batch_size("efficientnet", image_size),
            sampler=train_sampler,
            num_workers=0,
            pin_memory=PIN_MEMORY,
        )
    else:
        train_loader = DataLoader(
            datasets_dict["train"],
            batch_size=get_batch_size("efficientnet", image_size),
            shuffle=True,
            num_workers=0,
            pin_memory=PIN_MEMORY,
        )

    loaders = {
        "train": train_loader,
        "val": DataLoader(
            datasets_dict["val"],
            batch_size=get_batch_size("efficientnet", image_size),
            shuffle=False,
            num_workers=0,
            pin_memory=PIN_MEMORY,
        ),
        "test": DataLoader(
            datasets_dict["test"],
            batch_size=get_batch_size("efficientnet", image_size),
            shuffle=False,
            num_workers=0,
            pin_memory=PIN_MEMORY,
        ),
        "datasets": datasets_dict,
        "used_sampler": use_weighted_sampler,
    }
    return loaders


def build_efficientnet_model(pretrained: bool = True) -> tuple[nn.Module, bool]:
    try:
        model = timm.create_model(EFFICIENTNET_MODEL_NAME, pretrained=pretrained, num_classes=1)
        return model, pretrained
    except Exception as exc:
        print(f"Falha ao carregar pesos pretrained ({exc}). Usando random init.")
        model = timm.create_model(EFFICIENTNET_MODEL_NAME, pretrained=False, num_classes=1)
        return model, False


def train_one_epoch_classification(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> dict:
    model.train()
    total_loss = 0.0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    auc_score = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    return {
        "loss": total_loss / max(len(all_labels), 1),
        "auc": float(auc_score),
        "probs": all_probs,
        "labels": all_labels,
    }


@torch.no_grad()
def evaluate_classification(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict:
    model.eval()
    total_loss = 0.0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)

        total_loss += loss.item() * len(labels)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    auc_score = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    return {
        "loss": total_loss / max(len(all_labels), 1),
        "auc": float(auc_score),
        "probs": all_probs,
        "labels": all_labels,
    }


def run_efficientnet_experiment(dataset_info: dict) -> dict:
    dataset_key = dataset_info["requested_name"]
    image_size = dataset_info["image_size"]
    loaders = build_classification_loaders(dataset_info)

    model, used_pretrained = build_efficientnet_model(pretrained=USE_IMAGENET_PRETRAINING)
    model = model.to(DEVICE)

    train_manifest = dataset_info["manifests"]["train"]
    positive_count = max(int((train_manifest["binary_label"] == 1).sum()), 1)
    negative_count = max(int((train_manifest["binary_label"] == 0).sum()), 1)
    pos_weight = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=EFFICIENTNET_LR, weight_decay=EFFICIENTNET_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EFFICIENTNET_EPOCHS, eta_min=1e-6)

    history = {"train_loss": [], "train_auc": [], "val_loss": [], "val_auc": []}
    best_val_auc = -float("inf")
    best_state = None
    patience_counter = 0

    print(f"\n{'=' * 84}")
    print(f"EfficientNet-B0 | dataset={dataset_key} | resolved={dataset_info['resolved_name']} | size={image_size}")
    print(f"Weighted sampler no treino: {loaders['used_sampler']}")
    print(f"Pretrained ImageNet usado: {used_pretrained}")
    print(f"{'=' * 84}")

    for epoch in range(1, EFFICIENTNET_EPOCHS + 1):
        train_metrics = train_one_epoch_classification(model, loaders["train"], criterion, optimizer, DEVICE)
        val_metrics = evaluate_classification(model, loaders["val"], criterion, DEVICE)
        scheduler.step()

        history["train_loss"].append(train_metrics["loss"])
        history["train_auc"].append(train_metrics["auc"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_auc"].append(val_metrics["auc"])

        improved = val_metrics["auc"] > best_val_auc
        marker = " <<< melhor" if improved else ""
        print(
            f"Epoch {epoch:02d}/{EFFICIENTNET_EPOCHS} | "
            f"train loss={train_metrics['loss']:.4f} auc={train_metrics['auc']:.4f} | "
            f"val loss={val_metrics['loss']:.4f} auc={val_metrics['auc']:.4f}{marker}"
        )

        if improved:
            best_val_auc = val_metrics["auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EFFICIENTNET_PATIENCE:
                print(f"Early stopping na epoca {epoch}.")
                break

    model.load_state_dict(best_state)

    save_path = MODEL_DIR / f"efficientnet_b0_{dataset_key}.pt"
    torch.save(best_state, save_path)

    val_outputs = evaluate_classification(model, loaders["val"], criterion, DEVICE)
    test_outputs = evaluate_classification(model, loaders["test"], criterion, DEVICE)
    threshold_info = select_threshold_for_high_recall(
        val_outputs["probs"],
        val_outputs["labels"],
        target_recall=MELANOMA_RECALL_TARGET,
    )
    test_eval = compute_binary_metrics(
        test_outputs["probs"],
        test_outputs["labels"],
        threshold=threshold_info["threshold"],
    )

    roc_payload = compute_roc_payload(test_outputs["probs"], test_outputs["labels"])
    del model
    clear_device_cache()

    return {
        "model_family": "EfficientNet-B0",
        "dataset_info": dataset_info,
        "history": history,
        "model_path": save_path,
        "used_pretrained": used_pretrained,
        "val_outputs": val_outputs,
        "test_outputs": test_outputs,
        "threshold_info": threshold_info,
        "test_eval": test_eval,
        "roc_payload": roc_payload,
    }


print("Pipeline EfficientNet definido.")
        

Pipeline EfficientNet definido.


## 4. Modelo B: U-Net Multi-task

Correcao importante para os datasets com augmentation offline:

- imagens augmentadas nao possuem mascara augmentada correspondente no repositório
- por isso, o U-Net usa **loss de classificacao em todas as amostras**
- e aplica **loss de segmentacao apenas quando a mascara continua alinhada com a imagem**

Na pratica:

- `without_augmentation`: segmentacao ativa em todo o treino
- `with_augmentation_*`: segmentacao ativa nos originais; copias offline entram apenas na perda de classificacao
        

In [6]:
def load_mask_array(mask_path: Path) -> np.ndarray:
    mask = np.array(Image.open(mask_path).convert("L"))
    return (mask > MASK_THRESHOLD).astype(np.uint8)


def compute_mask_bbox(mask: np.ndarray) -> tuple[int, int, int, int]:
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        height, width = mask.shape[:2]
        return (0, 0, width, height)
    return (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)


def expand_bbox(
    bbox: tuple[int, int, int, int],
    width: int,
    height: int,
    margin_ratio: float = BBOX_MARGIN_RATIO,
) -> tuple[int, int, int, int]:
    x0, y0, x1, y1 = bbox
    box_w = max(x1 - x0, 1)
    box_h = max(y1 - y0, 1)
    margin_x = max(1, int(round(box_w * margin_ratio)))
    margin_y = max(1, int(round(box_h * margin_ratio)))
    return (
        max(0, x0 - margin_x),
        max(0, y0 - margin_y),
        min(width, x1 + margin_x),
        min(height, y1 + margin_y),
    )


def crop_array(array: np.ndarray, bbox: tuple[int, int, int, int]) -> np.ndarray:
    x0, y0, x1, y1 = bbox
    return array[y0:y1, x0:x1]


def pad_to_square(array: np.ndarray) -> np.ndarray:
    height, width = array.shape[:2]
    side = max(height, width)
    pad_top = (side - height) // 2
    pad_bottom = side - height - pad_top
    pad_left = (side - width) // 2
    pad_right = side - width - pad_left

    if array.ndim == 3:
        pad_width = ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0))
    else:
        pad_width = ((pad_top, pad_bottom), (pad_left, pad_right))

    return np.pad(array, pad_width=pad_width, mode="edge")


@lru_cache(maxsize=32768)
def build_processed_mask(image_id: str, target_size: int) -> np.ndarray:
    mask_path = MASK_DIR / f"{image_id}.png"
    mask = load_mask_array(mask_path)
    bbox = expand_bbox(compute_mask_bbox(mask), width=mask.shape[1], height=mask.shape[0])
    mask = crop_array(mask, bbox)
    mask = pad_to_square(mask)
    mask = cv2.resize(mask, (target_size, target_size), interpolation=cv2.INTER_NEAREST)
    return (mask > 0).astype(np.float32)


class MultitaskSkinLesionDataset(Dataset):
    def __init__(self, manifest: pd.DataFrame, image_size: int, mean: list[float], std: list[float], split_name: str) -> None:
        self.records = manifest.reset_index(drop=True)
        self.image_size = image_size
        self.split_name = split_name
        self.image_transform = transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ]
        )

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        row = self.records.iloc[idx]
        image = Image.open(row["export_path"]).convert("RGB")
        image_tensor = self.image_transform(image)

        mask_np = build_processed_mask(str(row["image_id"]), self.image_size)
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0).float()

        is_augmented = bool(row["is_augmented"]) if "is_augmented" in row.index else False
        mask_available = not (self.split_name == "train" and is_augmented)
        mask_available_tensor = torch.tensor(float(mask_available), dtype=torch.float32)
        label = torch.tensor(int(row["binary_label"]), dtype=torch.float32)

        return image_tensor, mask_tensor, label, mask_available_tensor


class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DownBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.30) -> None:
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(2)
        self.drop = nn.Dropout2d(dropout)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        skip = self.conv(x)
        out = self.drop(self.pool(skip))
        return skip, out


class UpBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int, dropout: float = 0.25) -> None:
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.drop = nn.Dropout2d(dropout)
        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.drop(x)
        return self.conv(x)


class UNetMultiTask(nn.Module):
    def __init__(self, in_channels: int = 3, enc_filters: list[int] | None = None, bridge_filters: int = 1024) -> None:
        super().__init__()
        enc_filters = enc_filters or UNET_ENCODER_FILTERS
        f = enc_filters

        self.enc1 = DownBlock(in_channels, f[0])
        self.enc2 = DownBlock(f[0], f[1])
        self.enc3 = DownBlock(f[1], f[2])
        self.enc4 = DownBlock(f[2], f[3])

        self.bridge = DoubleConv(f[3], bridge_filters)

        self.dec4 = UpBlock(bridge_filters, f[3], f[3])
        self.dec3 = UpBlock(f[3], f[2], f[2])
        self.dec2 = UpBlock(f[2], f[1], f[1])
        self.dec1 = UpBlock(f[1], f[0], f[0])

        self.seg_head = nn.Conv2d(f[0], 1, kernel_size=1)
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(bridge_filters, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        s1, x = self.enc1(x)
        s2, x = self.enc2(x)
        s3, x = self.enc3(x)
        s4, x = self.enc4(x)

        x = self.bridge(x)
        cls_logit = self.cls_head(x).squeeze(1)

        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        seg_logit = self.seg_head(x)
        return cls_logit, seg_logit


def build_multitask_loaders(dataset_info: dict) -> dict:
    manifests = dataset_info["manifests"]
    image_size = dataset_info["image_size"]
    mean = dataset_info["norm_mean"]
    std = dataset_info["norm_std"]

    datasets_dict = {
        split: MultitaskSkinLesionDataset(frame, image_size, mean, std, split)
        for split, frame in manifests.items()
    }

    loaders = {
        "train": DataLoader(
            datasets_dict["train"],
            batch_size=get_batch_size("unet_multitask", image_size),
            shuffle=True,
            num_workers=0,
            pin_memory=PIN_MEMORY,
        ),
        "val": DataLoader(
            datasets_dict["val"],
            batch_size=get_batch_size("unet_multitask", image_size),
            shuffle=False,
            num_workers=0,
            pin_memory=PIN_MEMORY,
        ),
        "test": DataLoader(
            datasets_dict["test"],
            batch_size=get_batch_size("unet_multitask", image_size),
            shuffle=False,
            num_workers=0,
            pin_memory=PIN_MEMORY,
        ),
        "datasets": datasets_dict,
    }
    return loaders


def dice_loss(pred_logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(pred_logits)
    intersection = (probs * targets).sum(dim=(2, 3))
    union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    dice = (2.0 * intersection + eps) / (union + eps)
    return 1.0 - dice.mean()


def combined_seg_loss(pred_logits: torch.Tensor, targets: torch.Tensor, bce_criterion: nn.Module, dice_weight: float = 0.5) -> torch.Tensor:
    bce = bce_criterion(pred_logits, targets)
    dice = dice_loss(pred_logits, targets)
    return (1.0 - dice_weight) * bce + dice_weight * dice


def masked_segmentation_loss(
    pred_logits: torch.Tensor,
    targets: torch.Tensor,
    mask_available: torch.Tensor,
    bce_criterion: nn.Module,
) -> torch.Tensor:
    valid_mask = mask_available > 0.5
    if int(valid_mask.sum().item()) == 0:
        return pred_logits.sum() * 0.0
    return combined_seg_loss(pred_logits[valid_mask], targets[valid_mask], bce_criterion)


def train_one_epoch_multitask(
    model: nn.Module,
    loader: DataLoader,
    cls_criterion: nn.Module,
    seg_bce_criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> dict:
    model.train()
    total_loss = cls_loss_total = seg_loss_total = 0.0
    all_probs, all_labels = [], []

    for images, masks, labels, mask_available in loader:
        images = images.to(device)
        masks = masks.to(device)
        labels = labels.to(device)
        mask_available = mask_available.to(device)

        optimizer.zero_grad()
        cls_logit, seg_logit = model(images)

        loss_cls = cls_criterion(cls_logit, labels)
        loss_seg = masked_segmentation_loss(seg_logit, masks, mask_available, seg_bce_criterion)
        loss = loss_cls + UNET_SEG_LOSS_WEIGHT * loss_seg

        loss.backward()
        optimizer.step()

        n = len(labels)
        total_loss += loss.item() * n
        cls_loss_total += loss_cls.item() * n
        seg_loss_total += loss_seg.item() * n

        probs = torch.sigmoid(cls_logit).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    auc_score = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    return {
        "loss": total_loss / max(len(all_labels), 1),
        "cls_loss": cls_loss_total / max(len(all_labels), 1),
        "seg_loss": seg_loss_total / max(len(all_labels), 1),
        "auc": float(auc_score),
        "probs": all_probs,
        "labels": all_labels,
    }


@torch.no_grad()
def evaluate_multitask(
    model: nn.Module,
    loader: DataLoader,
    cls_criterion: nn.Module,
    seg_bce_criterion: nn.Module,
    device: torch.device,
) -> dict:
    model.eval()
    total_loss = cls_loss_total = seg_loss_total = 0.0
    all_probs, all_labels = [], []
    seg_logits_list, seg_masks_list = [], []

    for images, masks, labels, mask_available in loader:
        images = images.to(device)
        masks = masks.to(device)
        labels = labels.to(device)
        mask_available = mask_available.to(device)

        cls_logit, seg_logit = model(images)
        loss_cls = cls_criterion(cls_logit, labels)
        loss_seg = masked_segmentation_loss(seg_logit, masks, mask_available, seg_bce_criterion)
        loss = loss_cls + UNET_SEG_LOSS_WEIGHT * loss_seg

        n = len(labels)
        total_loss += loss.item() * n
        cls_loss_total += loss_cls.item() * n
        seg_loss_total += loss_seg.item() * n

        probs = torch.sigmoid(cls_logit).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

        valid_mask = mask_available > 0.5
        if int(valid_mask.sum().item()) > 0:
            seg_logits_list.append(seg_logit[valid_mask].cpu())
            seg_masks_list.append(masks[valid_mask].cpu())

    auc_score = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0

    if len(seg_logits_list) > 0:
        seg_preds = (torch.sigmoid(torch.cat(seg_logits_list, dim=0)) > 0.5).float()
        seg_gt = torch.cat(seg_masks_list, dim=0)
        intersection = (seg_preds * seg_gt).sum()
        union = seg_preds.sum() + seg_gt.sum() - intersection
        iou = float((intersection + 1e-6) / (union + 1e-6))
        dice = float((2 * intersection + 1e-6) / (seg_preds.sum() + seg_gt.sum() + 1e-6))
    else:
        iou = float("nan")
        dice = float("nan")

    return {
        "loss": total_loss / max(len(all_labels), 1),
        "cls_loss": cls_loss_total / max(len(all_labels), 1),
        "seg_loss": seg_loss_total / max(len(all_labels), 1),
        "auc": float(auc_score),
        "iou": iou,
        "dice": dice,
        "probs": all_probs,
        "labels": all_labels,
    }


def run_unet_experiment(dataset_info: dict) -> dict:
    dataset_key = dataset_info["requested_name"]
    image_size = dataset_info["image_size"]
    loaders = build_multitask_loaders(dataset_info)

    model = UNetMultiTask(
        in_channels=3,
        enc_filters=UNET_ENCODER_FILTERS,
        bridge_filters=UNET_BRIDGE_FILTERS,
    ).to(DEVICE)

    train_manifest = dataset_info["manifests"]["train"]
    positive_count = max(int((train_manifest["binary_label"] == 1).sum()), 1)
    negative_count = max(int((train_manifest["binary_label"] == 0).sum()), 1)
    pos_weight = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=DEVICE)

    cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    seg_bce_criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=UNET_LR, weight_decay=UNET_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=UNET_EPOCHS, eta_min=1e-6)

    history = {
        "train_loss": [],
        "train_auc": [],
        "train_seg_loss": [],
        "val_loss": [],
        "val_auc": [],
        "val_iou": [],
        "val_dice": [],
    }
    best_val_auc = -float("inf")
    best_state = None
    patience_counter = 0

    print(f"\n{'=' * 84}")
    print(f"U-Net Multi-task | dataset={dataset_key} | resolved={dataset_info['resolved_name']} | size={image_size}")
    print(f"Train augmented copies: {dataset_info['has_augmented_train']}")
    print("Loss de segmentacao no treino e mascarada para copias offline augmentadas.")
    print(f"{'=' * 84}")

    for epoch in range(1, UNET_EPOCHS + 1):
        train_metrics = train_one_epoch_multitask(model, loaders["train"], cls_criterion, seg_bce_criterion, optimizer, DEVICE)
        val_metrics = evaluate_multitask(model, loaders["val"], cls_criterion, seg_bce_criterion, DEVICE)
        scheduler.step()

        history["train_loss"].append(train_metrics["loss"])
        history["train_auc"].append(train_metrics["auc"])
        history["train_seg_loss"].append(train_metrics["seg_loss"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_auc"].append(val_metrics["auc"])
        history["val_iou"].append(val_metrics["iou"])
        history["val_dice"].append(val_metrics["dice"])

        improved = val_metrics["auc"] > best_val_auc
        marker = " <<< melhor" if improved else ""
        print(
            f"Epoch {epoch:02d}/{UNET_EPOCHS} | "
            f"train loss={train_metrics['loss']:.4f} seg={train_metrics['seg_loss']:.4f} auc={train_metrics['auc']:.4f} | "
            f"val loss={val_metrics['loss']:.4f} auc={val_metrics['auc']:.4f} "
            f"iou={val_metrics['iou']:.4f} dice={val_metrics['dice']:.4f}{marker}"
        )

        if improved:
            best_val_auc = val_metrics["auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= UNET_PATIENCE:
                print(f"Early stopping na epoca {epoch}.")
                break

    model.load_state_dict(best_state)

    save_path = MODEL_DIR / f"unet_multitask_{dataset_key}.pt"
    torch.save(best_state, save_path)

    val_outputs = evaluate_multitask(model, loaders["val"], cls_criterion, seg_bce_criterion, DEVICE)
    test_outputs = evaluate_multitask(model, loaders["test"], cls_criterion, seg_bce_criterion, DEVICE)
    threshold_info = select_threshold_for_high_recall(
        val_outputs["probs"],
        val_outputs["labels"],
        target_recall=MELANOMA_RECALL_TARGET,
    )
    test_eval = compute_binary_metrics(
        test_outputs["probs"],
        test_outputs["labels"],
        threshold=threshold_info["threshold"],
    )

    roc_payload = compute_roc_payload(test_outputs["probs"], test_outputs["labels"])
    del model
    clear_device_cache()

    return {
        "model_family": "U-Net Multi-task",
        "dataset_info": dataset_info,
        "history": history,
        "model_path": save_path,
        "val_outputs": val_outputs,
        "test_outputs": test_outputs,
        "threshold_info": threshold_info,
        "test_eval": test_eval,
        "roc_payload": roc_payload,
    }


print("Pipeline U-Net Multi-task definido.")
        

Pipeline U-Net Multi-task definido.


## 5. Benchmark completo

Cada dataset eh usado para treinar os dois modelos. O benchmark registra:

- curva de treino
- AUC na validacao
- threshold escolhido com foco em sensibilidade
- metricas finais no test set
- IoU e Dice para o U-Net
        

In [ ]:
def summarize_run(result: dict) -> dict:
    dataset_info = result["dataset_info"]
    test_eval = result["test_eval"]
    threshold_info = result["threshold_info"]
    row = {
        "modelo": result["model_family"],
        "dataset_solicitado": dataset_info["requested_name"],
        "dataset_resolvido": dataset_info["resolved_name"],
        "image_size": dataset_info["image_size"],
        "target_recall_validacao": threshold_info["target_recall"],
        "target_atingido_validacao": threshold_info["target_hit"],
        "threshold": round(test_eval["threshold"], 4),
        "auc_test": round(test_eval["auc"], 4),
        "sensibilidade_test": round(test_eval["sensitivity"], 4),
        "especificidade_test": round(test_eval["specificity"], 4),
        "precisao_test": round(test_eval["precision"], 4),
        "f1_test": round(test_eval["f1"], 4),
        "f2_test": round(test_eval["f2"], 4),
        "acuracia_test": round(test_eval["accuracy"], 4),
        "tp_test": test_eval["tp"],
        "tn_test": test_eval["tn"],
        "fp_test": test_eval["fp"],
        "fn_test": test_eval["fn"],
        "selection_rule": threshold_info["selection_rule"],
    }

    if result["model_family"] == "U-Net Multi-task":
        row["iou_test"] = round(float(result["test_outputs"]["iou"]), 4)
        row["dice_test"] = round(float(result["test_outputs"]["dice"]), 4)
    else:
        row["iou_test"] = np.nan
        row["dice_test"] = np.nan

    return row


benchmark_runs = {}
benchmark_rows = []

for dataset_key, dataset_info in dataset_registry.items():
    eff_result = run_efficientnet_experiment(dataset_info)
    benchmark_runs[("EfficientNet-B0", dataset_key)] = eff_result
    benchmark_rows.append(summarize_run(eff_result))

    unet_result = run_unet_experiment(dataset_info)
    benchmark_runs[("U-Net Multi-task", dataset_key)] = unet_result
    benchmark_rows.append(summarize_run(unet_result))

benchmark_df = pd.DataFrame(benchmark_rows).sort_values(
    ["sensibilidade_test", "fn_test", "auc_test", "especificidade_test"],
    ascending=[False, True, False, False],
).reset_index(drop=True)

display(benchmark_df)
        


EfficientNet-B0 | dataset=with_augmentation_64x64 | resolved=with_augmentation_64x64 | size=64
Weighted sampler no treino: False
Pretrained ImageNet usado: True
Epoch 01/16 | train loss=5.0855 auc=0.7068 | val loss=4.7646 auc=0.6825 <<< melhor
Epoch 02/16 | train loss=2.4124 auc=0.8427 | val loss=4.3508 auc=0.7135 <<< melhor
Epoch 03/16 | train loss=1.6295 auc=0.8932 | val loss=3.5799 auc=0.7391 <<< melhor
Epoch 04/16 | train loss=1.2416 auc=0.9169 | val loss=3.2509 auc=0.7494 <<< melhor
Epoch 05/16 | train loss=0.9427 auc=0.9349 | val loss=2.6937 auc=0.7696 <<< melhor
Epoch 06/16 | train loss=0.7576 auc=0.9464 | val loss=2.3619 auc=0.7733 <<< melhor
Epoch 07/16 | train loss=0.6287 auc=0.9561 | val loss=2.2643 auc=0.7748 <<< melhor
Epoch 08/16 | train loss=0.4936 auc=0.9667 | val loss=2.2185 auc=0.7862 <<< melhor
Epoch 09/16 | train loss=0.4447 auc=0.9719 | val loss=2.0818 auc=0.7980 <<< melhor
Epoch 10/16 | train loss=0.3916 auc=0.9761 | val loss=1.8490 auc=0.8011 <<< melhor
Epoch 11

## 6. Curvas, ROC e comparacao visual
        

In [ ]:
palette = {
    "EfficientNet-B0": "#2b6cb0",
    "U-Net Multi-task": "#c53030",
}


def plot_validation_auc_curves(runs: dict, fig_path: Path | None = None) -> None:
    n_datasets = len(dataset_registry)
    fig, axes = plt.subplots(1, n_datasets, figsize=(5 * n_datasets, 4), sharey=True)
    if n_datasets == 1:
        axes = [axes]

    for ax, (dataset_key, dataset_info) in zip(axes, dataset_registry.items()):
        for model_family in ["EfficientNet-B0", "U-Net Multi-task"]:
            result = runs[(model_family, dataset_key)]
            history = result["history"]
            ax.plot(
                history["val_auc"],
                marker="o",
                linewidth=2,
                label=model_family,
                color=palette[model_family],
            )

        title = dataset_key
        if dataset_info["resolved_name"] != dataset_key:
            title += f"\n(resolvido em {dataset_info['resolved_name']})"
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_xlabel("Epoca")
        ax.set_ylabel("Val AUC")
        ax.set_ylim(0.4, 1.0)
        ax.grid(alpha=0.25)
        ax.legend()

    plt.suptitle("Curvas de validacao (AUC) por dataset", fontweight="bold")
    plt.tight_layout()
    if fig_path is not None:
        plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_roc_by_dataset(runs: dict, fig_path: Path | None = None) -> None:
    n_datasets = len(dataset_registry)
    fig, axes = plt.subplots(1, n_datasets, figsize=(5 * n_datasets, 4), sharex=True, sharey=True)
    if n_datasets == 1:
        axes = [axes]

    for ax, (dataset_key, dataset_info) in zip(axes, dataset_registry.items()):
        for model_family in ["EfficientNet-B0", "U-Net Multi-task"]:
            result = runs[(model_family, dataset_key)]
            roc_payload = result["roc_payload"]
            ax.plot(
                roc_payload["fpr"],
                roc_payload["tpr"],
                linewidth=2,
                color=palette[model_family],
                label=f"{model_family} (AUC={roc_payload['auc']:.3f})",
            )

        ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
        title = dataset_key
        if dataset_info["resolved_name"] != dataset_key:
            title += f"\n(resolvido em {dataset_info['resolved_name']})"
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.25)

    plt.suptitle("Curvas ROC no test set", fontweight="bold")
    plt.tight_layout()
    if fig_path is not None:
        plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()


comparison_plot_df = benchmark_df.copy()
comparison_plot_df["modelo_dataset"] = comparison_plot_df["modelo"] + " | " + comparison_plot_df["dataset_solicitado"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(
    data=comparison_plot_df,
    x="dataset_solicitado",
    y="sensibilidade_test",
    hue="modelo",
    palette=palette,
    ax=axes[0],
)
axes[0].set_title("Sensibilidade no test set", fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("Sensibilidade")
axes[0].set_ylim(0, 1)

sns.barplot(
    data=comparison_plot_df,
    x="dataset_solicitado",
    y="fn_test",
    hue="modelo",
    palette=palette,
    ax=axes[1],
)
axes[1].set_title("Falsos negativos no test set", fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("FN")

for ax in axes:
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(FIG_DIR / "benchmark_sensitivity_fn.png", dpi=150, bbox_inches="tight")
plt.show()

plot_validation_auc_curves(benchmark_runs, fig_path=FIG_DIR / "benchmark_val_auc_curves.png")
plot_roc_by_dataset(benchmark_runs, fig_path=FIG_DIR / "benchmark_roc_curves.png")
        

## 7. Analise de erros e visualizacao do melhor U-Net

O primeiro bloco mostra os falsos negativos do melhor modelo geral segundo o criterio clinico do projeto.

O segundo bloco mostra a segmentacao do melhor U-Net entre os seis experimentos.
        

In [ ]:
def denormalize_image(tensor: torch.Tensor, mean: list[float], std: list[float]) -> np.ndarray:
    image = tensor.detach().cpu().permute(1, 2, 0).numpy()
    image = image * np.array(std) + np.array(mean)
    return np.clip(image, 0.0, 1.0)


def show_false_negatives(result: dict, n_show: int = 6, fig_path: Path | None = None) -> None:
    dataset_info = result["dataset_info"]
    if result["model_family"] == "EfficientNet-B0":
        test_dataset = build_classification_loaders(dataset_info)["datasets"]["test"]
    else:
        test_dataset = build_multitask_loaders(dataset_info)["datasets"]["test"]

    test_eval = result["test_eval"]
    labels = np.asarray(test_eval["labels"])
    preds = np.asarray(test_eval["preds"])
    probs = np.asarray(test_eval["probs"])

    fn_indices = np.where((labels == 1) & (preds == 0))[0][:n_show]
    if len(fn_indices) == 0:
        print("Nenhum falso negativo encontrado para este modelo.")
        return

    fig, axes = plt.subplots(1, len(fn_indices), figsize=(3 * len(fn_indices), 3))
    if len(fn_indices) == 1:
        axes = [axes]

    for ax, idx in zip(axes, fn_indices):
        record = test_dataset.records.iloc[idx]
        image = Image.open(record["export_path"]).convert("RGB")
        ax.imshow(np.array(image))
        ax.set_title(
            f"p={probs[idx]:.2f}\ntrue=MEL\npred=Non-MEL",
            fontsize=8,
            color="#c53030",
            fontweight="bold",
        )
        ax.axis("off")

    plt.suptitle(
        f"Falsos negativos | {result['model_family']} | {dataset_info['requested_name']}",
        fontweight="bold",
    )
    plt.tight_layout()
    if fig_path is not None:
        plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()


def load_unet_from_result(result: dict) -> nn.Module:
    model = UNetMultiTask(
        in_channels=3,
        enc_filters=UNET_ENCODER_FILTERS,
        bridge_filters=UNET_BRIDGE_FILTERS,
    )
    state_dict = torch.load(result["model_path"], map_location="cpu")
    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    model.eval()
    return model


@torch.no_grad()
def visualize_unet_predictions(result: dict, n_melanoma: int = 3, n_non_melanoma: int = 3, fig_path: Path | None = None) -> None:
    dataset_info = result["dataset_info"]
    model = load_unet_from_result(result)
    test_dataset = build_multitask_loaders(dataset_info)["datasets"]["test"]

    mel_idx = [i for i in range(len(test_dataset)) if int(test_dataset.records.iloc[i]["binary_label"]) == 1][:n_melanoma]
    nonmel_idx = [i for i in range(len(test_dataset)) if int(test_dataset.records.iloc[i]["binary_label"]) == 0][:n_non_melanoma]
    indices = mel_idx + nonmel_idx

    n = len(indices)
    fig, axes = plt.subplots(4, n, figsize=(3 * n, 12))
    if n == 1:
        axes = np.array(axes).reshape(4, 1)

    row_labels = ["Imagem", "Mascara GT", "Mascara Predita", "Overlay"]
    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=10)

    for col, idx in enumerate(indices):
        image_tensor, mask_tensor, label_tensor, _mask_available = test_dataset[idx]
        cls_logit, seg_logit = model(image_tensor.unsqueeze(0).to(DEVICE))
        cls_prob = torch.sigmoid(cls_logit).item()
        pred_mask = (torch.sigmoid(seg_logit) > 0.5).float().squeeze().cpu().numpy()

        image_np = denormalize_image(image_tensor, dataset_info["norm_mean"], dataset_info["norm_std"])
        gt_mask = mask_tensor.squeeze().numpy()

        lbl_name = "MEL" if int(label_tensor.item()) == 1 else "Non-MEL"
        color = "#c53030" if int(label_tensor.item()) == 1 else "#2b6cb0"

        axes[0, col].imshow(image_np)
        axes[0, col].set_title(f"{lbl_name}\np={cls_prob:.2f}", color=color, fontsize=8, fontweight="bold")
        axes[0, col].axis("off")

        axes[1, col].imshow(gt_mask, cmap="gray", vmin=0, vmax=1)
        axes[1, col].axis("off")

        axes[2, col].imshow(pred_mask, cmap="gray", vmin=0, vmax=1)
        intersection = (pred_mask * gt_mask).sum()
        union = pred_mask.sum() + gt_mask.sum() - intersection
        iou_sample = float((intersection + 1e-6) / (union + 1e-6))
        axes[2, col].set_title(f"IoU={iou_sample:.2f}", fontsize=8)
        axes[2, col].axis("off")

        overlay = (image_np * 255).astype(np.uint8).copy()
        contours, _ = cv2.findContours((pred_mask * 255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, contours, -1, (255, 120, 0), 2)
        axes[3, col].imshow(overlay)
        axes[3, col].axis("off")

    plt.suptitle(
        f"Melhor U-Net | {dataset_info['requested_name']} | {dataset_info['resolved_name']}",
        fontweight="bold",
    )
    plt.tight_layout()
    if fig_path is not None:
        plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()


best_overall_row = benchmark_df.sort_values(
    ["sensibilidade_test", "fn_test", "auc_test", "especificidade_test"],
    ascending=[False, True, False, False],
).iloc[0]
best_overall_result = benchmark_runs[(best_overall_row["modelo"], best_overall_row["dataset_solicitado"])]

best_unet_row = benchmark_df[benchmark_df["modelo"] == "U-Net Multi-task"].sort_values(
    ["sensibilidade_test", "fn_test", "auc_test", "especificidade_test"],
    ascending=[False, True, False, False],
).iloc[0]
best_unet_result = benchmark_runs[(best_unet_row["modelo"], best_unet_row["dataset_solicitado"])]

print("Melhor modelo geral segundo o criterio clinico:")
display(pd.DataFrame([best_overall_row]))
show_false_negatives(best_overall_result, fig_path=FIG_DIR / "best_model_false_negatives.png")

print("Melhor execucao do U-Net:")
display(pd.DataFrame([best_unet_row]))
visualize_unet_predictions(best_unet_result, fig_path=FIG_DIR / "best_unet_predictions.png")
        

## 8. Conclusoes
        

In [ ]:
ranking_df = benchmark_df.sort_values(
    ["sensibilidade_test", "fn_test", "auc_test", "especificidade_test"],
    ascending=[False, True, False, False],
).reset_index(drop=True)

best_row = ranking_df.iloc[0]
best_family = best_row["modelo"]
best_requested_dataset = best_row["dataset_solicitado"]
best_resolved_dataset = best_row["dataset_resolvido"]

family_best_df = (
    benchmark_df.sort_values(
        ["sensibilidade_test", "fn_test", "auc_test", "especificidade_test"],
        ascending=[False, True, False, False],
    )
    .groupby("modelo", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
display(family_best_df)

print("Conclusoes:")
print(
    f"- O melhor resultado geral foi {best_family} treinado em `{best_requested_dataset}` "
    f"(resolvido em `{best_resolved_dataset}`), com sensibilidade={best_row['sensibilidade_test']:.4f}, "
    f"FN={int(best_row['fn_test'])} e AUC={best_row['auc_test']:.4f}."
)
print(
    f"- O ranking final priorizou deteccao de melanoma: primeiro sensibilidade, depois numero de falsos negativos, "
    f"seguido de AUC e especificidade."
)
print(
    f"- O threshold operacional foi escolhido na validacao buscando sensibilidade >= {MELANOMA_RECALL_TARGET:.2f}. "
    f"Isto deixa a comparacao alinhada com o risco clinico de perder melanomas."
)
print(
    f"- A comparacao high-resolution foi fixada explicitamente em `with_augmentation_224x224` "
    f"(com fallback para `with_augmentation` caso essa branch equivalente seja a que exista no repositório)."
)
print(
    "- Para o U-Net Multi-task, as copias offline augmentadas entram na loss de classificacao, enquanto a loss "
    "de segmentacao e aplicada apenas nas amostras cuja mascara continua alinhada."
)
print(
    "- Na decisao final do projeto, priorize o experimento com maior sensibilidade e menor FN, mesmo que haja "
    "pequena perda de especificidade, porque o custo clinico do falso negativo e maior."
)
        